<a href="https://colab.research.google.com/github/nattynatasha-gif/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Network Data Clean-Up

In [7]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
print("Raw rows:", len(txns))
clean_df, log = clean_agent_data(txns, agents)
print("Clean rows:", len(clean_df))

Raw rows: 612
Clean rows: 600


In [8]:
audit = pd.DataFrame([
    {"issue": "Duplicate txn_id", "dimension": "Uniqueness", "column": "txn_id", "rows_affected": 12},
    {"issue": "Inconsistent txn_type casing/whitespace", "dimension": "Consistency", "column": "txn_type", "rows_affected": 612},
    {"issue": "Missing / non-numeric amount", "dimension": "Completeness / Validity", "column": "amount", "rows_affected": 20},
    {"issue": "Slash-separated dates", "dimension": "Validity", "column": "txn_date", "rows_affected": 5},
    {"issue": "IQR outliers (incl. 250000)", "dimension": "Accuracy", "column": "amount", "rows_affected": 53},
    {"issue": "Unknown agent_id AG099", "dimension": "Consistency", "column": "agent_id", "rows_affected": 8},
])
audit

,issue,dimension,column,rows_affected
0,Duplicate txn_id,Uniqueness,txn_id,12
1,Inconsistent txn_type casing/whitespace,Consistency,txn_type,612
2,Missing / non-numeric amount,Completeness / Validity,amount,20
3,Slash-separated dates,Validity,txn_date,5
4,IQR outliers (incl. 250000),Accuracy,amount,53
5,Unknown agent_id AG099,Consistency,agent_id,8


In [9]:
print("Clean rows:", len(clean_df))
print("Duplicate txn_id left:", clean_df["txn_id"].duplicated().sum())
print("txn_type values:", sorted(clean_df["txn_type"].unique()))
print("Unknown agents:", (clean_df["county"] == "Unknown").sum())
print("amount_imputed:", clean_df["amount_imputed"].sum())
print("is_outlier:", clean_df["is_outlier"].sum())

Clean rows: 600
Duplicate txn_id left: 0
txn_type values: ['Deposit', 'Withdrawal']
Unknown agents: 7
amount_imputed: 20
is_outlier: 53


In [10]:
county_summary = (clean_df.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"), n_txns=("txn_id", "count"), avg_value=("amount", "mean"))
    .round(0))
county_summary

total_value  n_txns  avg_value
county  txn_type                                  
Kiambu  Deposit        334737.0      43     7785.0
        Withdrawal      77050.0      27     2854.0
Kisumu  Deposit        436420.0      74     5898.0
        Withdrawal      76374.0      37     2064.0
Mombasa Deposit        148986.0      72     2069.0
        Withdrawal     368592.0      61     6042.0
Nairobi Deposit        543590.0     118     4607.0
        Withdrawal     162036.0      62     2613.0
Nakuru  Deposit        184890.0      60     3082.0
        Withdrawal     328594.0      39     8426.0
Unknown Deposit         25279.0       6     4213.0
        Withdrawal       1877.0       1     1877.0

In [11]:
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)
weekly = clean_df.pivot_table(index="week", columns="county", values="amount", aggfunc="sum", fill_value=0)
weekly

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [12]:
import os
os.makedirs("outputs", exist_ok=True)
clean_df.drop(columns=["week"], errors="ignore").to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved. Columns:", list(clean_df.columns))

Saved. Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'week']


## Data Quality Report

### 1. Completeness
20 amounts were missing or invalid. They were imputed with the median and flagged with `amount_imputed = True`.

### 2. Uniqueness
12 duplicate txn_id rows were removed. Final row count is 600.

### 3. Consistency
txn_type was standardised to exactly “Deposit” and “Withdrawal”. 8 transactions referenced unknown agent AG099 and were kept with county = “Unknown”.

### 4. Validity / Accuracy
Amounts converted to numeric. Dates with “/” were fixed. 53 amounts flagged as IQR outliers (including 250000 values). These are kept but marked `is_outlier = True` for business review.

### 5. Timeliness
All dates fall between March and April 2026. No future or stale dates.

### 6. Integrity
After left join every row has a valid txn_id, standardised txn_type, numeric amount, ISO date and a county (or “Unknown”).

### Privacy (Kenya Data Protection Act 2019)
If the extract also contained customer phone numbers, those would be personal data. They must be collected for a lawful purpose, minimised, protected by access controls and encryption, retained only as long as necessary, and deleted or anonymised when no longer needed.